In [ ]:
! pip install -q kaggle


In [ ]:
from google.colab import files
files.upload()

In [ ]:
! mkdir -p ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle competitions download -c plant-seedlings-classification

In [ ]:
# Replace 'dataset-name.zip' with the name of the file that was downloaded
! unzip plant-seedlings-classification.zip

In [ ]:
import os
import glob
from PIL import Image
import numpy as np
import tensorflow as tf

## Load and preprocess images

### Subtask:
Load the images from the collected paths and preprocess them (e.g., resize, normalize) for use in a machine learning model.

**Reasoning**:
Define a function to load and preprocess images, then apply it to the collected image paths and convert the results to NumPy arrays.

In [ ]:
import os
import glob
from PIL import Image
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

train_dir = 'train'
test_dir = 'test'

train_image_paths = []
train_image_labels = []

test_image_paths = []
test_image_labels = [] # Initialize test_image_labels as well

# Load training data
for class_name in os.listdir(train_dir):
    class_dir = os.path.join(train_dir, class_name)
    if os.path.isdir(class_dir):
        for image_name in os.listdir(class_dir):
            if image_name.endswith('.png'): # Assuming images are in PNG format
                image_path = os.path.join(class_dir, image_name)
                train_image_paths.append(image_path)
                train_image_labels.append(class_name)

# Split training data into training and validation sets
train_image_paths, val_image_paths, train_image_labels, val_image_labels = train_test_split(
    train_image_paths, train_image_labels, test_size=0.2, random_state=42, stratify=train_image_labels
)


# Load testing data
# Check if the test directory has subdirectories for labels or just image files
if os.path.isdir(os.path.join(test_dir, os.listdir(test_dir)[0])) and not os.path.isfile(os.path.join(test_dir, os.listdir(test_dir)[0])):
    # Test directory has subdirectories for labels
    for class_name in os.listdir(test_dir):
        class_dir = os.path.join(test_dir, class_name)
        if os.path.isdir(class_dir):
            for image_name in os.listdir(class_dir):
                if image_name.endswith('.png'): # Assuming images are in PNG format
                    image_path = os.path.join(class_dir, image_name)
                    test_image_paths.append(image_path)
                    test_image_labels.append(class_name)
else:
    # Test directory contains only image files
    for image_name in os.listdir(test_dir):
        if image_name.endswith('.png'): # Assuming images are in PNG format
            image_path = os.path.join(test_dir, image_name)
            test_image_paths.append(image_path)
            # For test set without labels, we can append None or a placeholder
            test_image_labels.append(None) # Or leave it as an empty list if labels are not available


print(f"Number of training images: {len(train_image_paths)}")
print(f"Number of validation images: {len(val_image_paths)}")
print(f"Number of testing images: {len(test_image_paths)}")

def load_and_preprocess_image(image_path, target_size):
    """Loads and preprocesses an image."""
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    img_array = np.array(img) / 255.0
    return img_array

target_size = (128, 128)

train_images = [load_and_preprocess_image(path, target_size) for path in train_image_paths]
val_images = [load_and_preprocess_image(path, target_size) for path in val_image_paths] # Preprocess validation images
test_images = [load_and_preprocess_image(path, target_size) for path in test_image_paths]

train = np.array(train_images)
val = np.array(val_images) # Convert validation images to numpy array
test = np.array(test_images)

print(f"Shape of training images array: {train.shape}")
print(f"Shape of validation images array: {val.shape}") # Print shape of validation images
print(f"Shape of testing images array: {test.shape}")

## Prepare labels

### Subtask:
Convert the categorical string labels into numerical format (e.g., using one-hot encoding or label encoding).

**Reasoning**:
The subtask is to convert the categorical string labels into numerical format using label encoding and one-hot encoding. I will import the necessary modules, apply label encoding to the training labels, then apply one-hot encoding to training, validation, and testing numerical labels.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform training labels
train_numerical_labels = label_encoder.fit_transform(train_image_labels)

# Transform validation labels using the encoder fitted on training labels
val_numerical_labels = label_encoder.transform(val_image_labels) # Transform validation labels

# Apply one-hot encoding to training numerical labels
train_one_hot_labels = to_categorical(train_numerical_labels)

# Apply one-hot encoding to validation numerical labels
val_one_hot_labels = to_categorical(val_numerical_labels, num_classes=len(label_encoder.classes_)) # One-hot encode validation labels

# Filter out None values from test_image_labels before transforming
test_image_labels_filtered = [label for label in test_image_labels if label is not None]

# Transform testing labels using the encoder fitted on training labels
# Only transform if there are labels in the test set
if test_image_labels_filtered:
    test_numerical_labels = label_encoder.transform(test_image_labels_filtered)
    # Apply one-hot encoding to testing numerical labels
    test_one_hot_labels = to_categorical(test_numerical_labels, num_classes=len(label_encoder.classes_))
else:
    test_one_hot_labels = None # Set to None if no test labels


print("Training labels have been converted to numerical and one-hot encoded.")
print("Validation labels have been converted to numerical and one-hot encoded.") # Print status for validation labels
if test_one_hot_labels is not None:
    print("Testing labels have been converted to numerical and one-hot encoded.")
else:
    print("Skipping testing label conversion as test data does not have labels.")

## Build the Model

### Subtask:
Define the architecture of a convolutional neural network (CNN) suitable for image classification.

**Reasoning**:
Define a sequential Keras model with convolutional, pooling, flatten, and dense layers for image classification. The output layer will have a number of neurons equal to the number of unique classes and use a softmax activation function.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as np # Import numpy

# Define the number of classes
num_classes = len(label_encoder.classes_)

# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=train.shape[1:]), # Use train.shape[1:] for input shape
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

## Compile the Model

### Subtask:
Configure the model for training by specifying the optimizer, loss function, and metrics.

**Reasoning**:
Compile the sequential Keras model using the Adam optimizer, categorical crossentropy loss function, and accuracy as a metric.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Model compiled successfully.")

## Train the Model

### Subtask:
Train the compiled model using the preprocessed training data and labels, and validate using the validation set.

**Reasoning**:
Train the compiled Keras model using the `fit()` method with the preprocessed training image data and one-hot encoded training labels. Use the validation data and labels for monitoring performance during training.

In [ ]:
# Train the model
history = model.fit(train, train_one_hot_labels, epochs=10, validation_data=(val, val_one_hot_labels))

## Evaluate the Model

### Subtask:
Evaluate the trained model's performance on the test dataset.

**Reasoning**:
Evaluate the trained Keras model using the `evaluate()` method with the preprocessed test image data and one-hot encoded test labels (if available).

In [ ]:
# Evaluate the model only if there are test labels
if test_one_hot_labels is not None:
    loss, accuracy = model.evaluate(test, test_one_hot_labels)

    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
else:
    print("Skipping model evaluation as test data does not have labels.")

## Make Predictions

### Subtask:
Use the trained model to make predictions on the test dataset.

**Reasoning**:
Use the trained Keras model's `predict()` method to generate predictions for the preprocessed test image data. Convert the predicted probabilities to class labels and then back to original class names.

In [ ]:
# Make predictions on the test set
predictions = model.predict(test)

# Convert predictions from probabilities to class labels
predicted_classes_numerical = np.argmax(predictions, axis=1)

# Convert numerical labels back to class names
predicted_classes_names = label_encoder.inverse_transform(predicted_classes_numerical)

print("Predictions generated for the test set.")

## Analyze Predictions

### Subtask:
Analyze the distribution of predicted classes and the confidence of the predictions.

**Reasoning**:
Calculate and display the distribution of the predicted classes to understand which classes the model is predicting most often. Also, analyze the distribution of the prediction confidence scores to assess the model's certainty in its predictions.

In [ ]:
# Get the distribution of predicted classes
predicted_class_distribution = submission_df['species'].value_counts()

print("Distribution of Predicted Classes:")
display(predicted_class_distribution)

# Optional: Visualize the distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
predicted_class_distribution.plot(kind='bar')
plt.title('Distribution of Predicted Classes')
plt.xlabel('Plant Species')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Get the confidence score for the predicted class for each image
predicted_confidence = np.max(predictions, axis=1)

print("Prediction Confidence Statistics:")
display(pd.Series(predicted_confidence).describe())

# Optional: Visualize the distribution of prediction confidence
plt.figure(figsize=(10, 6))
plt.hist(predicted_confidence, bins=20, edgecolor='black')
plt.title('Distribution of Prediction Confidence')
plt.xlabel('Confidence Score')
plt.ylabel('Number of Images')
plt.show()

## Visualize Sample Predictions

### Subtask:
Display a few sample images from the test set along with their predicted labels.

**Reasoning**:
Randomly select a few images from the test set for each predicted class and display them along with their predicted labels to visually inspect the model's performance.

In [ ]:
import random
import matplotlib.pyplot as plt

# Get the unique predicted classes
unique_predicted_classes = np.unique(predicted_classes_names)

# Define the number of samples to display per class
num_samples_per_class = 3 # You can change this number

plt.figure(figsize=(15, 10))

for i, class_name in enumerate(unique_predicted_classes):
    # Get the indices of images predicted as this class
    indices = [j for j, label in enumerate(predicted_classes_names) if label == class_name]

    # Randomly select a few indices
    sample_indices = random.sample(indices, min(len(indices), num_samples_per_class))

    # Display the sample images
    for k, sample_idx in enumerate(sample_indices):
        plt.subplot(len(unique_predicted_classes), num_samples_per_class, i * num_samples_per_class + k + 1)
        # Display the original image (assuming test_images is a list of numpy arrays)
        plt.imshow(test_images[sample_idx])
        plt.title(f"Predicted: {class_name}")
        plt.axis('off')

plt.tight_layout()
plt.show()

## Save the Model

### Subtask:
Save the trained model to a file for future use.

**Reasoning**:
Save the trained Keras model to a file so it can be loaded and used later without retraining.

In [ ]:
# Save the trained model
model_save_path = 'plant_seedling_model.keras'
model.save(model_save_path)

print(f"Model saved successfully to {model_save_path}")

## Create Submission File

### Subtask:
Create a submission file in the specified format (e.g., CSV) with filenames and predicted labels.

**Reasoning**:
Create a pandas DataFrame with the test filenames and their corresponding predicted class names. Save this DataFrame to a CSV file in the format required for submission.

In [ ]:
import pandas as pd
import os

# Extract just the filenames from the full paths
test_filenames = [os.path.basename(path) for path in test_image_paths]

# Create a DataFrame with filenames and predicted labels
submission_df = pd.DataFrame({'file': test_filenames, 'species': predicted_classes_names})

# Save the DataFrame to a CSV file
submission_file_path = 'submission.csv'
submission_df.to_csv(submission_file_path, index=False)

print(f"Submission file created successfully at: {submission_file_path}")
display(submission_df.head())